# Visual-Language Assistant with Moondream2 and OpenVINO

[Moondream2](https://huggingface.co/vikhyatk/moondream2) is a small (2B parameters) vision-language model designed to run efficiently on edge devices. Despite its compact size, it supports a wide range of vision-language tasks including image captioning, visual question answering, object detection, pointing, and OCR.

In this notebook, we demonstrate how to convert Moondream2 to OpenVINO IR format with INT4 weight compression and run inference for multiple tasks. All inference runs on OpenVINO for optimized performance on Intel hardware.


#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Select Model](#Select-Model)
- [Convert and Optimize Model](#Convert-and-Optimize-Model)
- [Run Model Inference](#Run-Model-Inference)
  - [Select Inference Device](#Select-Inference-Device)
  - [Load Model](#Load-Model)
  - [Image Captioning](#Image-Captioning)
  - [Visual Question Answering](#Visual-Question-Answering)
  - [Object Detection](#Object-Detection)
- [Interactive Demo with Gradio](#Interactive-Demo-with-Gradio)


## Prerequisites

[back to top ⬆️](#Table-of-contents:)

Install required dependencies.

In [ ]:
%pip install -q "torch>=2.4" "torchvision" "Pillow" "gradio>=4.36" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q -U "openvino>=2025.0.0" "openvino-tokenizers>=2025.0.0" "nncf>=2.14.0"
%pip install -q -U "optimum-intel[openvino]>=1.22.0" "transformers>=4.45.0" "ipywidgets"

In [ ]:
from pathlib import Path
import requests

if not Path("cmd_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py")
    open("cmd_helper.py", "w").write(r.text)

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w").write(r.text)

In [ ]:
import ipywidgets as widgets

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("moondream2-vision.ipynb")

## Select Model

[back to top ⬆️](#Table-of-contents:)

Moondream2 has been updated through several revisions. We use a recent stable revision for best quality.

In [ ]:
model_id = "vikhyatk/moondream2"
model_revision = "2025-01-09"

base_model_path = Path("moondream2")

print(f"Model: {model_id} (revision: {model_revision})")

## Convert and Optimize Model

[back to top ⬆️](#Table-of-contents:)

Moondream2 is a PyTorch model. OpenVINO supports PyTorch models via conversion to OpenVINO Intermediate Representation (IR). We use the [Optimum CLI](https://huggingface.co/docs/optimum/intel/openvino/export) to export the model with INT4 weight compression, which significantly reduces memory footprint and improves inference latency on CPU.

In [ ]:
to_compress = widgets.Checkbox(value=True, description="Compress to INT4")
to_compress

In [ ]:
from cmd_helper import optimum_cli

model_path = base_model_path / ("INT4" if to_compress.value else "FP16")

additional_args = {
    "trust-remote-code": "",
    "task": "image-text-to-text",
}

if to_compress.value:
    additional_args["weight-format"] = "int4"
    additional_args["group-size"] = "64"
else:
    additional_args["weight-format"] = "fp16"

if not model_path.exists():
    optimum_cli(model_id, model_path, additional_args)
else:
    print(f"Model already converted: {model_path}")

## Run Model Inference

[back to top ⬆️](#Table-of-contents:)

OpenVINO integration with Optimum Intel provides ready-to-use API for model inference. We use `OVModelForVisualCausalLM` for loading and running the converted model.

### Select Inference Device

[back to top ⬆️](#Table-of-contents:)

In [ ]:
from notebook_utils import device_widget

device = device_widget(default="AUTO", exclude=["NPU"])

device

### Load Model

[back to top ⬆️](#Table-of-contents:)

In [ ]:
from transformers import AutoProcessor, AutoTokenizer, TextStreamer
from optimum.intel.openvino import OVModelForVisualCausalLM

processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
ov_model = OVModelForVisualCausalLM.from_pretrained(model_path, device=device.value, trust_remote_code=True)

print(f"Model loaded on {device.value}")

### Image Captioning

[back to top ⬆️](#Table-of-contents:)

Let's start with image captioning. We load a sample image and ask the model to describe it.

In [ ]:
from PIL import Image
from io import BytesIO


def load_image(image_url: str) -> Image.Image:
    """Load an image from a URL."""
    response = requests.get(image_url)
    return Image.open(BytesIO(response.content)).convert("RGB")


sample_image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/300px-PNG_transparency_demonstration_1.png"
image = load_image(sample_image_url)
display(image)

In [ ]:
caption_prompt = "Describe this image in detail."

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": caption_prompt},
        ],
    }
]

text = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=[text], images=[image], return_tensors="pt")

streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

print(f"Question: {caption_prompt}")
print("Answer: ", end="")
ov_model.generate(**inputs, do_sample=False, max_new_tokens=256, streamer=streamer);

### Visual Question Answering

[back to top ⬆️](#Table-of-contents:)

Now let's ask specific questions about an image.

In [ ]:
nature_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/b/b6/Image_created_with_a_mobile_phone.png/1280px-Image_created_with_a_mobile_phone.png"
nature_image = load_image(nature_url)
display(nature_image)

questions = [
    "What is the main subject of this image?",
    "What colors are dominant in this scene?",
    "What time of day does this appear to be?",
]

for question in questions:
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": question},
            ],
        }
    ]
    text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=[text], images=[nature_image], return_tensors="pt")
    output = ov_model.generate(**inputs, do_sample=False, max_new_tokens=128)
    answer = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"Q: {question}")
    print(f"A: {answer}\n")

### Object Detection

[back to top ⬆️](#Table-of-contents:)

Moondream2 can detect objects and return bounding boxes. We ask the model to locate specific objects in the image.

In [ ]:
detect_prompt = "List all objects you can see in this image with their approximate positions (left, center, right, top, bottom)."

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": detect_prompt},
        ],
    }
]

text = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=[text], images=[nature_image], return_tensors="pt")

print(f"Question: {detect_prompt}")
print("Answer: ", end="")
ov_model.generate(**inputs, do_sample=False, max_new_tokens=256, streamer=streamer);

## Interactive Demo with Gradio

[back to top ⬆️](#Table-of-contents:)

Launch an interactive interface to chat with Moondream2 about any image.

In [ ]:
import gradio as gr


def answer_question(image: Image.Image, question: str) -> str:
    """Run VQA on an uploaded image."""
    if image is None:
        return "Please upload an image."
    if not question.strip():
        return "Please enter a question."

    image = image.convert("RGB")
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": question},
            ],
        }
    ]
    text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt")
    output = ov_model.generate(**inputs, do_sample=False, max_new_tokens=256)
    answer = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return answer


demo = gr.Interface(
    fn=answer_question,
    inputs=[
        gr.Image(type="pil", label="Upload Image"),
        gr.Textbox(label="Question", placeholder="Ask something about the image...", lines=2),
    ],
    outputs=gr.Textbox(label="Answer", lines=4),
    title="Moondream2 Visual Assistant with OpenVINO",
    description="Upload an image and ask questions about it. Powered by Moondream2 (2B) optimized with OpenVINO.",
    examples=[
        [sample_image_url, "What is shown in this image?"],
        [nature_url, "Describe the scene in detail."],
    ],
    allow_flagging="never",
)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)

In [ ]:
# # Cleanup: uncomment to remove downloaded model files
# import shutil
# if base_model_path.exists():
#     shutil.rmtree(base_model_path)
#     print(f"Removed {base_model_path}")